## 1. Import Libraries

In [148]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from tqdm.notebook import tqdm
import time
import re
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Bank Data

In [149]:
# Load data bank dari CSV
df_banks = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/ojk_cfs_bpr_konvensional.csv')

print(f"Total banks loaded: {len(df_banks)}")
print(f"Provinsi available: {len(df_banks['Provinsi'].unique())} provinsi")
df_banks.head(10)

Total banks loaded: 1863
Provinsi available: 33 provinsi


,Provinsi,Kabupaten/Kota,Nama Bank,Kode Bank
0,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Cikarang Raharja,600007
1,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Dana Multi Guna,600036
2,Provinsi Jawa Barat,Kab. Bekasi,PT. BPR Siwa Raharja Utama,600070
3,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Binadana Makmur,600098
4,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Sentral Mandiri,600103
5,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Antar Guna,600108
6,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Artha Sentana Hardja,600131
7,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Karya Kurniautama,600135
8,Provinsi Jawa Barat,Kab. Bekasi,PT. BPR Artaprima Danajasa,600137
9,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Olympindo Primadana,600139


## 3. Configuration

In [150]:
BASE_URL = "https://cfs.ojk.go.id/cfs/ReportViewerForm.aspx"
MONTH = "12"  # Fixed: Desember
PERIOD_TYPE = "R"

REPORT_TYPES = {
    "BPK-900-000001": "Laporan Neraca",
    "BPK-900-000002": "Laporan Laba Rugi",
    "BPK-900-000004": "Laporan Informasi Lainnya"
}

COLUMN_ORDER = [
    'Tahun', 'Bulan', 'Longitude', 'Latitude', 'Nama_BPR', 
    'Kabupaten_Kota', 'Provinsi', 'Kode_Bank',
    # Posisi Keuangan (Laporan Neraca)
    'Total_Aset', 
    'a. Kepada BPR',
    'b. Kepada Bank Umum',
    'c. Kepada non bank – pihak terkait',
    'd. Kepada non bank – pihak tidak terkait',
    'Cadangan_Kerugian_Penurunan_Nilai',
    'Jumlah_Kredit',
    'Tabungan', 
    'Deposito', 
    'Total_Liabilitas', 
    'Laba_tahun_berjalan', 
    'Total_Ekuitas',
    'Agunan_yang_Diambil_Alih',
    # Laba Rugi
    'Jumlah_Pendapatan_Bunga', 
    'Jumlah_Pendapatan_Operasional', 
    'Jumlah_Beban_Operasional',
    'Laba_Rugi_Operasional',
    # Rasio Keuangan (Laporan Informasi Lainnya)
    'NPL_Gross', 
    'NPL_Neto', 
    'ROA', 
    'BOPO', 
    'NIM', 
    'LDR', 
    'KPMM',
    'Cash_Ratio'
]

print(" Configuration loaded")

 Configuration loaded


## 4. Helper Functions

In [151]:
def clean_number(value):
    """Convert string number dengan format Indonesia ke float"""
    if pd.isna(value) or value == '' or value == 'NaN':
        return 0
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value = str(value).strip()
    value = re.sub(r'[^\d.,()\-]', '', value)
    
    if '(' in value and ')' in value:
        value = '-' + value.replace('(', '').replace(')', '')
    
    value = value.replace(',', '')
    
    try:
        return float(value)
    except:
        return 0

def find_main_table(tables):
    """Cari tabel utama yang berisi data laporan"""
    best_table = None
    max_score = 0
    best_idx = -1
    
    for idx, table in enumerate(tables):
        try:
            html_string = str(table)
            df = pd.read_html(StringIO(html_string))[0]
            
            score = 0
            
            # Check if table contains ratio keywords
            text_content = df.to_string().lower()
            ratio_keywords = ['npl', 'roa', 'bopo', 'ldr', 'kpmm', 'nim', 'cash ratio']
            ratio_matches = sum(1 for kw in ratio_keywords if kw in text_content)
            if ratio_matches >= 3:
                score += ratio_matches * 100
            
            # Size scoring
            if df.shape[0] > 20:
                score += df.shape[0]
            if 2 <= df.shape[1] <= 10:
                score += 50
            
            # Numeric content scoring
            numeric_count = 0
            for col in df.columns:
                try:
                    nums = pd.to_numeric(df[col], errors='coerce')
                    if (nums > 1000).sum() > 5:
                        numeric_count += 1
                except:
                    pass
            
            if numeric_count > 0:
                score += numeric_count * 30
            
            if score > max_score:
                max_score = score
                best_table = df
                best_idx = idx
                
        except Exception as e:
            continue
    
    return best_table, best_idx

def find_value_in_row(df, keywords, col_index=2):
    """Cari nilai di tabel berdasarkan keyword"""
    for col in df.columns[:2]:
        for keyword in keywords:
            mask = df[col].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            if mask.any():
                idx = df[mask].index[0]
                if col_index < len(df.columns):
                    return clean_number(df.iloc[idx, col_index])
                return 0
    return 0

print("Helper functions defined")

Helper functions defined


## 5. Extraction & Parsing Functions

In [152]:
def extract_table(bank_code_number, bank_code, year, report_type, timeout=15, force_table_index=None):
    """Ekstrak tabel dari laporan"""
    params = {
        'BankCodeNumber': bank_code_number,
        'BankCode': bank_code,
        'Month': MONTH,
        'Year': str(year),
        'FinancialReportPeriodTypeCode': PERIOD_TYPE,
        'FinancialReportTypeCode': report_type
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        tables = soup.find_all('table')
        
        if not tables:
            return None
        
        # If force_table_index specified, use that (untuk ratio table)
        if force_table_index is not None and force_table_index < len(tables):
            html_string = str(tables[force_table_index])
            forced_table = pd.read_html(StringIO(html_string))[0]
            return forced_table
        
        main_table, idx = find_main_table(tables)
        return main_table
        
    except Exception as e:
        return None

def parse_posisi_keuangan(df, year):
    """Parse Laporan Posisi Keuangan"""
    data = {}
    
    # Special handling for Total_Aset - Skip rows containing "Tetap"
    total_aset = 0
    for col in df.columns[:2]:
        for keyword in ['Jumlah Aset', 'JUMLAH AKTIVA']:
            mask = df[col].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            if mask.any():
                for idx in df[mask].index:
                    row_text = str(df.iloc[idx, col])
                    # Skip if row contains "Tetap" (e.g., "Jumlah Aset Tetap")
                    if 'Tetap' not in row_text and 'tetap' not in row_text:
                        if 2 < len(df.columns):
                            total_aset = clean_number(df.iloc[idx, 2])
                            break
                if total_aset != 0:
                    break
        if total_aset != 0:
            break
    data['Total_Aset'] = total_aset
    
    fields_mapping = {
        'a. Kepada BPR': [['a. Kepada BPR', 'Kepada BPR', 'b. Pada BPR', 'Pada BPR'], 2],
        'b. Kepada Bank Umum': [['b. Kepada Bank Umum', 'Kepada Bank Umum', 'a. Pada bank umum', 'Pada bank umum'], 2],
        'c. Kepada non bank – pihak terkait': [['c. Kepada non bank – pihak terkait', 'c. Kepada non bank - pihak terkait', 'a. Pihak Terkait', 'Pihak Terkait'], 2],
        'd. Kepada non bank – pihak tidak terkait': [['d. Kepada non bank – pihak tidak terkait', 'd. Kepada non bank - pihak tidak terkait', 'b. Pihak Tidak Terkait'], 2],
        'Tabungan': [['a. Tabungan', 'a.    Tabungan'], 2],
        'Deposito': [['b. Deposito', 'b.    Deposito'], 2],
        'Total_Liabilitas': [['Total Liabilitas', 'Jumlah Kewajiban'], 2],
        'Laba_tahun_berjalan': [['b. Tahun Berjalan', 'l.  Laba/rugi tahun berjalan', 'l. Laba/rugi tahun berjalan'], 2],
        'Agunan_yang_Diambil_Alih': [['Agunan yang Diambil Alih'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        value = find_value_in_row(df, keywords, col_idx)
        data[field] = value
    
    # Special handling Cadangan_Kerugian_Penurunan_Nilai
    # 2010-2012: No "Jumlah Kredit" row, directly search for Penyisihan
    # 2013+: Search AFTER "Jumlah Kredit" to skip first Penyisihan
    cadangan_val = 0
    
    if 2010 <= year <= 2012:
        # 2010-2012: Directly search for Penyisihan (only one occurrence after kredit list)
        # Use flexible matching to handle different whitespace variations
        for row_idx in range(len(df)):
            for col in df.columns[:2]:
                cell_text = str(df.iloc[row_idx, col])
                # Check if row contains both "Penyisihan" and "Aktiva Produktif"
                if 'Penyisihan' in cell_text and 'Aktiva Produktif' in cell_text:
                    if 2 < len(df.columns):
                        cadangan_val = clean_number(df.iloc[row_idx, 2])
                        break
                # Also check for alternative format
                elif 'Penyisihan Penghapusan' in cell_text and '-/-' in cell_text:
                    if 2 < len(df.columns):
                        cadangan_val = clean_number(df.iloc[row_idx, 2])
                        break
            if cadangan_val != 0:
                break
    else:
        # 2013+: Find "Jumlah Kredit" first to mark position
        kredit_row_idx = -1
        for row_idx in range(len(df)):
            for col in df.columns[:2]:
                if 'Jumlah Kredit' in str(df.iloc[row_idx, col]):
                    kredit_row_idx = row_idx
                    break
            if kredit_row_idx >= 0:
                break
        
        # Search for Penyisihan AFTER Kredit row
        start_row = kredit_row_idx if kredit_row_idx >= 0 else 0
        cadangan_keywords = ['Penyisihan Kerugian -/-', 'Cadangan Kerugian Penurunan Nilai']
        for row_idx in range(start_row, len(df)):
            for col in df.columns[:2]:
                for keyword in cadangan_keywords:
                    if keyword in str(df.iloc[row_idx, col]):
                        if 2 < len(df.columns):
                            cadangan_val = clean_number(df.iloc[row_idx, 2])
                            break
                if cadangan_val != 0:
                    break
            if cadangan_val != 0:
                break
    
    data['Cadangan_Kerugian_Penurunan_Nilai'] = cadangan_val
    
    # Calculate Jumlah_Kredit
    jumlah_kredit_calculated = (
        data.get('a. Kepada BPR', 0) + 
        data.get('b. Kepada Bank Umum', 0) + 
        data.get('c. Kepada non bank – pihak terkait', 0) + 
        data.get('d. Kepada non bank – pihak tidak terkait', 0)
    )
    jumlah_kredit_direct = find_value_in_row(df, ['Jumlah Kredit yang Diberikan', 'Jumlah Kredit', 'Total Kredit'], 2)
    data['Jumlah_Kredit'] = jumlah_kredit_direct if jumlah_kredit_direct != 0 else jumlah_kredit_calculated
    
    # Special handling Total_Ekuitas untuk tahun 2010-2012
    if 2010 <= year <= 2012:
        modal_dasar = find_value_in_row(df, ['a. Modal Dasar', 'a.  Modal Dasar'], 2)
        modal_belum_disetor = find_value_in_row(df, ['b. Modal yang belum disetor', 'b.  Modal yang belum disetor  -/-', 'Modal yang belum disetor -/-'], 2)
        laba_ditahan = find_value_in_row(df, ['k. Laba yang ditahan', 'k.  Laba yang ditahan'], 2)
        laba_tahun_berjalan = data.get('Laba_tahun_berjalan', 0)
        data['Total_Ekuitas'] = modal_dasar + modal_belum_disetor + laba_ditahan + laba_tahun_berjalan
    else:
        data['Total_Ekuitas'] = find_value_in_row(df, ['Total Ekuitas', 'Jumlah Ekuitas', 'Total Modal', 'JUMLAH MODAL'], 2)
    
    return data

def parse_laba_rugi(df, year):
    """Parse Laporan Laba Rugi"""
    data = {}
    
    # Keywords berbeda untuk 2010-2012 vs 2013+
    if 2010 <= year <= 2012:
        fields_mapping = {
            'Jumlah_Pendapatan_Bunga': [['-  Bunga', 'Bunga', 'Jumlah Pendapatan Bunga'], 2],
            'Jumlah_Pendapatan_Operasional': [['Jumlah Pendapatan Operasional', 'JUMLAH PENDAPATAN OPERASIONAL'], 2],
            'Jumlah_Beban_Operasional': [['Jumlah Beban Operasional', 'JUMLAH BEBAN OPERASIONAL'], 2],
            'Laba_Rugi_Operasional': [['Laba/Rugi sebelum Pajak Penghasilan (PPh)', 'LABA (RUGI) TAHUN BERJALAN SEBELUM PAJAK PENGHASILAN'], 2],
        }
    else:
        fields_mapping = {
            'Jumlah_Pendapatan_Bunga': [['Jumlah Pendapatan Bunga', '-  Bunga', 'Bunga'], 2],
            'Jumlah_Pendapatan_Operasional': [['JUMLAH PENDAPATAN OPERASIONAL', 'Jumlah Pendapatan Operasional'], 2],
            'Jumlah_Beban_Operasional': [['JUMLAH BEBAN OPERASIONAL', 'Jumlah Beban Operasional'], 2],
            'Laba_Rugi_Operasional': [['LABA (RUGI) BERSIH', 'Laba (Rugi) Bersih', 'LABA (RUGI) OPERASIONAL'], 2],
        }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        value = find_value_in_row(df, keywords, col_idx)
        data[field] = value
    
    return data

def parse_kualitas_aset(df, year):
    """Parse Laporan Kualitas Aset Produktif - ENHANCED VERSION"""
    data = {}
    
    # SPECIAL HANDLING for 2010-2012: Limited ratio availability
    if 2010 <= year <= 2012:
        # For 2010-2012, only extract ratios that are EXPLICITLY shown in the report
        # Do NOT calculate NPL from KL+D+M - that's absolute values, not ratio
        
        ratio_mappings = {
            'NPL_Neto': ['Non Performing Loan (NPL) Neto', '(NPL) Neto', 'NPL (neto)', '4.    NPL net (%)', 'NPL net (%)', 'a. NPL net'],
            'NPL_Gross': ['Non Performing Loan (NPL) Gross', 'NPL) Gross'],
            'ROA': ['Return on Assets (ROA)', '(ROA)', 'ROA', '7.    Return on Asset / ROA (%)', 'Return on Asset / ROA (%)', 'd. ROA'],
            'BOPO': ['Biaya Operasional terhadap Pendapatan Operasional (BOPO)', '(BOPO)', 'BOPO', 'g. BOPO'],
            'NIM': ['Net Interest Margin (NIM)', '(NIM)'],
            'LDR': ['Loan to Deposit Ratio (LDR)', '(LDR)', 'LDR', '6.    Loan to Deposit Ratio / LDR (%)', 'Loan to Deposit Ratio / LDR (%)', 'c. LDR'],
            'KPMM': ['Kewajiban Penyediaan Modal Minimum (KPMM)', '(KPMM)', 'KPMM', '5.    Rasio KPMM (%)', 'Rasio KPMM (%)', 'b. KPMM'],
            'Cash_Ratio': ['Cash Ratio', 'Cash Ratio', 'h. Cash Ratio']
        }
        
        for field_name, keywords in ratio_mappings.items():
            found = False
            for row_idx in range(len(df)):
                row_text = ' '.join([str(df.iloc[row_idx, col]) for col in range(len(df.columns))]).lower()
                
                for keyword in keywords:
                    if keyword.lower() in row_text:
                        # Get last column value (ratio column)
                        for col_idx in range(len(df.columns) - 1, -1, -1):
                            val_str = str(df.iloc[row_idx, col_idx])
                            if val_str not in ['NaN', 'nan', ''] and any(c.isdigit() for c in val_str):
                                val = clean_number(val_str)
                                # Ratio values should be between 0-1000%
                                if 0 < val < 1000:
                                    data[field_name] = val
                                    found = True
                                    break
                        if found:
                            break
                if found:
                    break
    
    else:
        # 2013+ format: Extract ratios from table
        if df.shape[1] >= 5:
            # Look for "Rasio-Rasio" section
            for row_idx in range(len(df)):
                for col_idx in range(len(df.columns)):
                    cell_value = str(df.iloc[row_idx, col_idx])
                    if 'Rasio-Rasio' in cell_value or 'rasio' in cell_value.lower():
                        ratio_mappings = {
                            'Cash Ratio': 'Cash_Ratio',
                            'KPMM': 'KPMM',
                            'NPL gross': 'NPL_Gross',
                            'NPL net': 'NPL_Neto',
                            'LDR': 'LDR',
                            'NIM': 'NIM',
                            'ROA': 'ROA',
                            'BOPO': 'BOPO',
                            'KAP': 'NPL_Gross',
                        }
                        
                        for check_row in range(row_idx + 1, min(row_idx + 11, len(df))):
                            row_text = ' '.join([str(df.iloc[check_row, i]) for i in range(len(df.columns))])
                            
                            for keyword, field_name in ratio_mappings.items():
                                if keyword.lower() in row_text.lower():
                                    for col in range(len(df.columns) - 1, -1, -1):
                                        val = str(df.iloc[check_row, col])
                                        if val not in ['NaN', 'nan', ''] and any(c.isdigit() for c in val):
                                            try:
                                                num_val = clean_number(val)
                                                if 0 < num_val < 1000:
                                                    if field_name not in data or data[field_name] == 0:
                                                        data[field_name] = num_val
                                                    break
                                            except:
                                                pass
                        break
        
        # Fallback method for 2013+
        if len(data) == 0:
            rasio_keywords = {
                'NPL_Neto': ['Non Performing Loan (NPL) Neto', '(NPL) Neto', 'NPL (neto)', '4.    NPL net (%)', 'NPL net (%)', 'a. NPL net'],
                'NPL_Gross': ['Non Performing Loan (NPL) Gross', 'NPL) Gross'],
                'ROA': ['Return on Assets (ROA)', '(ROA)', 'ROA', '7.    Return on Asset / ROA (%)', 'Return on Asset / ROA (%)', 'd. ROA'],
                'BOPO': ['Biaya Operasional terhadap Pendapatan Operasional (BOPO)', '(BOPO)', 'BOPO', 'g. BOPO'],
                'NIM': ['Net Interest Margin (NIM)', '(NIM)'],
                'LDR': ['Loan to Deposit Ratio (LDR)', '(LDR)', 'LDR', '6.    Loan to Deposit Ratio / LDR (%)', 'Loan to Deposit Ratio / LDR (%)', 'c. LDR'],
                'KPMM': ['Kewajiban Penyediaan Modal Minimum (KPMM)', '(KPMM)', 'KPMM', '5.    Rasio KPMM (%)', 'Rasio KPMM (%)', 'b. KPMM',],
                'Cash_Ratio': ['Cash Ratio', 'Cash Ratio', 'h. Cash Ratio']
            }
            
            for field, keywords in rasio_keywords.items():
                found = False
                for row_idx in range(len(df)):
                    for col_idx in range(len(df.columns)):
                        cell_value = str(df.iloc[row_idx, col_idx]).lower()
                        for keyword in keywords:
                            if keyword.lower() in cell_value:
                                for next_col in range(len(df.columns)):
                                    val = str(df.iloc[row_idx, next_col])
                                    if val not in ['NaN', 'nan', ''] and any(c.isdigit() for c in val):
                                        cleaned_val = clean_number(val)
                                        if 0 < cleaned_val < 1000:
                                            data[field] = cleaned_val
                                            found = True
                                            break
                                if found:
                                    break
                        if found:
                            break
                    if found:
                        break
    
    # Fill missing ratios
    for field in ['NPL_Neto', 'NPL_Gross', 'ROA', 'BOPO', 'NIM', 'LDR', 'KPMM', 'Cash_Ratio']:
        if field not in data:
            data[field] = 0
    
    return data

print(" Extraction & Parsing functions defined")

 Extraction & Parsing functions defined


## 6. Main Scraper Class

In [153]:
class BPRScraper:
    """Main scraper class dengan ThreadPoolExecutor"""
    
    def __init__(self):
        self.results = []
        
    def scrape_single_bank(self, bank_row, year):
        """Scrape data untuk satu bank di satu tahun"""
        bank_code_number = str(bank_row['Kode Bank'])
        bank_code = bank_row['Nama Bank']
        
        result = {
            'Tahun': int(year),
            'Bulan': MONTH,
            'Nama_BPR': bank_code,
            'Kabupaten_Kota': bank_row['Kabupaten/Kota'],
            'Provinsi': bank_row['Provinsi'],
            'Kode_Bank': bank_code_number,
            'Longitude': None,
            'Latitude': None,
            'status': 'failed'
        }
        
        try:
            # Laporan Posisi Keuangan
            # 2010-2012: Only 2 tables, use find_main_table
            # 2013+: 40+ tables, use force_table_index=37
            if 2010 <= year <= 2012:
                df_posisi = extract_table(bank_code_number, bank_code, year, "BPK-900-000001")
            else:
                df_posisi = extract_table(bank_code_number, bank_code, year, "BPK-900-000001", force_table_index=37)
            
            if df_posisi is not None:
                posisi_data = parse_posisi_keuangan(df_posisi, year)
                result.update(posisi_data)
            
            time.sleep(0.1)
            
            # Laporan Laba Rugi
            if 2010 <= year <= 2012:
                df_laba = extract_table(bank_code_number, bank_code, year, "BPK-900-000002")
            else:
                df_laba = extract_table(bank_code_number, bank_code, year, "BPK-900-000002", force_table_index=37)
            
            if df_laba is not None:
                laba_data = parse_laba_rugi(df_laba, year)
                result.update(laba_data)
            
            time.sleep(0.1)
            
            # Laporan Informasi Lainnya
            if 2010 <= year <= 2012:
                df_kualitas = extract_table(bank_code_number, bank_code, year, "BPK-900-000004")
            else:
                df_kualitas = extract_table(bank_code_number, bank_code, year, "BPK-900-000004", force_table_index=37)
            
            if df_kualitas is not None:
                kualitas_data = parse_kualitas_aset(df_kualitas, year)
                result.update(kualitas_data)
            
            result['status'] = 'success'
            
        except Exception as e:
            result['error'] = str(e)
        
        return result
    
    def run(self, df_banks_filtered, years, max_workers=4):
        """Run scraping dengan ThreadPoolExecutor"""
        tasks = []
        for _, bank_row in df_banks_filtered.iterrows():
            for year in years:
                tasks.append((bank_row, year))
        
        total_tasks = len(tasks)
        results = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.scrape_single_bank, task[0], task[1]): task for task in tasks}
            
            with tqdm(total=total_tasks, desc="Scraping", unit="task") as pbar:
                for future in as_completed(futures):
                    try:
                        result = future.result()
                        results.append(result)
                    except Exception as e:
                        print(f"Task failed: {e}")
                    pbar.update(1)
        
        return results

print("BPRScraper class defined")


BPRScraper class defined


## 7. USER CONFIGURATION

**Edit parameter di bawah ini:**

In [154]:
# ============================================================================
# EDIT PARAMETER DI SINI
# ============================================================================

# Rentang tahun
YEAR_START = 2010
YEAR_END = 2018

# Pilih provinsi:
# - ['SEMUA'] untuk scrape semua provinsi
# - Atau list provinsi spesifik: ['Provinsi Jawa Barat', 'Provinsi Papua Barat']
SELECTED_PROVINCES = ['SEMUA']

# Jumlah thread workers (4-8 optimal)
MAX_WORKERS = 8

# Nama file output
OUTPUT_FILENAME = 'bpr_financial_data'

# ============================================================================

print("Configuration set:")
print(f"   Tahun: {YEAR_START} - {YEAR_END}")
print(f"   Provinsi: {SELECTED_PROVINCES}")
print(f"   Workers: {MAX_WORKERS}")


Configuration set:
   Tahun: 2010 - 2018
   Provinsi: ['SEMUA']
   Workers: 8


## 8. RUN SCRAPING

**Jalankan cell ini untuk memulai scraping:**

In [155]:
# Prepare data
years = list(range(YEAR_START, YEAR_END + 1))

# Filter banks by province
if 'SEMUA' in SELECTED_PROVINCES:
    filtered_banks = df_banks.copy()
else:
    filtered_banks = df_banks[df_banks['Provinsi'].isin(SELECTED_PROVINCES)].copy()

total_tasks = len(filtered_banks) * len(years)

print("="*70)
print(" STARTING SCRAPING")
print("="*70)
print(f"Tahun: {YEAR_START} - {YEAR_END} ({len(years)} tahun)")
print(f"Bulan: Desember (Fixed)")
print(f"Provinsi: {', '.join(SELECTED_PROVINCES)}")
print(f"Jumlah Bank: {len(filtered_banks)}")
print(f"Total Tasks: {total_tasks}")
print(f"Workers: {MAX_WORKERS}")
print("="*70)
print()

# Run scraping
start_time = time.time()
scraper = BPRScraper()
results = scraper.run(filtered_banks, years, max_workers=MAX_WORKERS)
elapsed_time = time.time() - start_time

# Process results
df_results = pd.DataFrame(results)

# Add missing columns
for col in COLUMN_ORDER:
    if col not in df_results.columns:
        df_results[col] = 0

# Reorder columns
available_cols = [col for col in COLUMN_ORDER if col in df_results.columns]
df_results = df_results[available_cols]

# Statistics
success_count = (df_results['status'] == 'success').sum() if 'status' in df_results.columns else len(df_results)
failed_count = total_tasks - success_count

print("\n" + "="*70)
print(" SCRAPING COMPLETED")
print("="*70)
print(f"Total Tasks: {total_tasks}")
print(f"Success: {success_count} ({success_count/total_tasks*100:.1f}%)")
print(f"Failed: {failed_count} ({failed_count/total_tasks*100:.1f}%)")
print(f"Elapsed Time: {elapsed_time:.2f} seconds")
print(f"Average: {elapsed_time/total_tasks:.2f} sec/task")
print("="*70)

# Remove status column
df_export = df_results.drop(columns=['status'], errors='ignore')

# SKIP COLUMNS WITH 0 VALUES - Replace 0 with empty string for ratio columns
ratio_cols = ['NPL_Neto', 'NPL_Gross', 'ROA', 'BOPO', 'NIM', 'LDR', 'KPMM', 'Cash_Ratio']
for col in ratio_cols:
    if col in df_export.columns:
        df_export[col] = df_export[col].replace(0, '')

# Save files
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_file = f"{OUTPUT_FILENAME}_{timestamp}.csv"
df_export.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"\nSaved: {csv_file}")

try:
    excel_file = f"{OUTPUT_FILENAME}_{timestamp}.xlsx"
    df_export.to_excel(excel_file, index=False, engine='openpyxl')
    print(f" Saved: {excel_file}")
except Exception as e:
    print(f"Excel save failed: {e}")

print("\n Done!")

 STARTING SCRAPING
Tahun: 2010 - 2018 (9 tahun)
Bulan: Desember (Fixed)
Provinsi: SEMUA
Jumlah Bank: 1863
Total Tasks: 16767
Workers: 8



Scraping:   0%|          | 0/16767 [00:00<?, ?task/s]


 SCRAPING COMPLETED
Total Tasks: 16767
Success: 16767 (100.0%)
Failed: 0 (0.0%)
Elapsed Time: 38708.88 seconds
Average: 2.31 sec/task

Saved: bpr_financial_data_20260102_052059.csv
 Saved: bpr_financial_data_20260102_052059.xlsx

 Done!


## 9. Preview Results

In [163]:
print("Sample Data (First 5 rows):")
df_export[df_export['Tahun'] == 2018].iloc[:,5:]

Sample Data (First 5 rows):


,Kabupaten_Kota,Provinsi,Kode_Bank,Total_Aset,a. Kepada BPR,b. Kepada Bank Umum,c. Kepada non bank – pihak terkait,d. Kepada non bank – pihak tidak terkait,Cadangan_Kerugian_Penurunan_Nilai,Jumlah_Kredit,...,Jumlah_Beban_Operasional,Laba_Rugi_Operasional,NPL_Gross,NPL_Neto,ROA,BOPO,NIM,LDR,KPMM,Cash_Ratio
11,Kab. Bekasi,Provinsi Jawa Barat,600007,89826074.0,0.0,0.0,1818288.0,75364405.0,4919362.0,77182693.0,...,8731664.0,449688.0,21.1,20.58,0.96,94.18,,88.06,11.06,9.12
18,Kab. Bekasi,Provinsi Jawa Barat,600036,23833381.0,0.0,0.0,479653.0,15335009.0,113915.0,15814662.0,...,4035509.0,291217.0,7.01,11.61,1.51,93.09,,70.11,26.23,5.85
29,Kab. Bekasi,Provinsi Jawa Barat,600070,26208400.0,0.0,0.0,19161.0,19607210.0,143570.0,19626371.0,...,2844272.0,1295636.0,3.14,4.3,6.31,75.19,,81.11,34.98,30.32
39,Kab. Bekasi,Provinsi Jawa Barat,600098,4346547.0,0.0,0.0,110230.0,2295716.0,259447.0,2405946.0,...,1336583.0,-695840.0,14.11,11.28,,204.93,,49.79,58.56,69.61
48,Kab. Bekasi,Provinsi Jawa Barat,600103,11081989.0,0.0,0.0,47398.0,9912650.0,780401.0,9960048.0,...,3803346.0,-475366.0,14.95,17.48,,107.27,,101.94,12.04,2.53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16744,Kab. Manokwari,Provinsi Papua Barat,602643,616937742.0,0.0,0.0,2852317.0,514457295.0,17323995.0,517309612.0,...,34987991.0,9130134.0,32.73,29.74,2.18,86.76,,94.43,15.0,16.51
16746,Kab. Manokwari,Provinsi Papua Barat,602738,44287346.0,0.0,0.0,384196.0,25507278.0,239352.0,25891474.0,...,3335045.0,751472.0,2.93,6.34,1.93,89.73,,59.45,24.95,41.25
16750,Kab. Manokwari,Provinsi Papua Barat,602742,36759501.0,0.0,0.0,88180.0,30705427.0,155159.0,30793607.0,...,3474794.0,-1163670.0,,,5.13,130.82,,85.48,12.9,11.11
16765,Kota Sorong,Provinsi Papua Barat,601289,13397917.0,0.0,0.0,0.0,5313111.0,255944.0,5313111.0,...,2622035.0,-1371202.0,13.74,16.6,,194.91,,40.3,27.75,56.52


In [164]:
df_export

,Tahun,Bulan,Longitude,Latitude,Nama_BPR,Kabupaten_Kota,Provinsi,Kode_Bank,Total_Aset,a. Kepada BPR,...,Jumlah_Beban_Operasional,Laba_Rugi_Operasional,NPL_Gross,NPL_Neto,ROA,BOPO,NIM,LDR,KPMM,Cash_Ratio
0,2010,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,28995036.0,2688563.0,...,6330019.0,-605372.0,,,,,,,,
1,2012,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,33100385.0,3794995.0,...,6498079.0,843929.0,,,,,,,,
2,2011,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,31323896.0,1732617.0,...,5788072.0,1101720.0,,,,,,,,
3,2015,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,54264220.0,0.0,...,8245408.0,-1237552.0,16.61,20.76,,110.03,,80.07,12.26,5.32
4,2017,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,81490572.0,0.0,...,8147389.0,2492460.0,11.75,14.0,3.89,81.61,,82.31,12.87,12.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16762,2015,12,None,None,PT Bank Perekonomian Rakyat Menara Cendrawasih...,Kota Sorong,Provinsi Papua Barat,601289,17608057.0,0.0,...,1152062.0,982822.0,0.62,0.68,5.56,64.46,,76.05,24.12,37.82
16763,2016,12,None,None,PT Bank Perekonomian Rakyat Menara Cendrawasih...,Kota Sorong,Provinsi Papua Barat,601289,19901061.0,0.0,...,4478119.0,458878.0,0.23,0.24,2.63,92.73,,86.4,22.54,30.17
16764,2017,12,None,None,PT Bank Perekonomian Rakyat Menara Cendrawasih...,Kota Sorong,Provinsi Papua Barat,601289,7418980.0,0.0,...,3700703.0,-872096.0,18.73,29.31,,122.9,,67.33,52.64,58.21
16765,2018,12,None,None,PT Bank Perekonomian Rakyat Menara Cendrawasih...,Kota Sorong,Provinsi Papua Barat,601289,13397917.0,0.0,...,2622035.0,-1371202.0,13.74,16.6,,194.91,,40.3,27.75,56.52


In [170]:
dg = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Full1924.csv')

df = pd.concat([df_export, dg], ignore_index=True)
df = df.reset_index(drop=True)
df = df.sort_values(by=['Tahun'])
df = df.reset_index(drop=True)

In [171]:
df.to_excel('/Users/mraffyzeidan/Learning/xcap/Ful1024.xlsx', index=False)

In [172]:
df.to_csv('/Users/mraffyzeidan/Learning/xcap/Full1024.csv', index=False, encoding='utf-8-sig')

In [167]:
print("Summary Statistics (Transposed):")
df.describe().T

Summary Statistics (Transposed):


,count,mean,std,min,25%,50%,75%,max
Tahun,27945.0,2.017000e+03,4.320571e+00,2.010000e+03,2013.00,2017.0,2021.00,2.024000e+03
Total_Aset,22442.0,9.165178e+09,1.222069e+11,0.000000e+00,12316494.75,27607245.0,75586176.50,1.120546e+13
a. Kepada BPR,22442.0,5.361690e+07,2.986983e+09,0.000000e+00,0.00,0.0,0.00,2.855531e+11
b. Kepada Bank Umum,22442.0,1.426857e+06,1.399597e+07,0.000000e+00,0.00,0.0,0.00,1.058449e+09
c. Kepada non bank – pihak terkait,22442.0,1.341395e+08,2.691273e+09,-8.300000e+01,10286.00,139941.0,551219.25,2.091464e+11
d. Kepada non bank – pihak tidak terkait,22442.0,6.488541e+09,9.435575e+10,0.000000e+00,8275880.50,19125885.5,53086891.75,8.964319e+12
Cadangan_Kerugian_Penurunan_Nilai,22442.0,2.012534e+08,2.222596e+09,0.000000e+00,161025.50,411398.5,1253895.25,1.021784e+11
Jumlah_Kredit,22442.0,6.477018e+09,9.478352e+10,-1.167000e+03,9067262.25,20140914.0,55443389.75,9.125527e+12
Tabungan,22442.0,2.017808e+09,2.373072e+10,0.000000e+00,378408.75,3509448.5,12592242.75,1.597118e+12
Deposito,22442.0,4.426503e+09,5.766437e+10,0.000000e+00,521500.00,8227212.0,30213219.00,4.254124e+12


## 10. Data Quality Check

**Check ratio values:**

In [145]:
ratio_cols = ['NPL_Neto', 'NPL_Gross', 'ROA', 'BOPO', 'NIM', 'LDR', 'KPMM', 'Cash_Ratio']

print(" Ratio Data Check:")
print("="*50)
for col in ratio_cols:
    non_zero = (df_export[col] != 0).sum()
    total = len(df_export)
    pct = (non_zero / total * 100) if total > 0 else 0
    print(f"{col:20s}: {non_zero:3d}/{total:3d} ({pct:5.1f}%) have values")

print("\n Ratio Statistics:")
df_export[ratio_cols].describe()

 Ratio Data Check:
NPL_Neto            :  18/ 18 (100.0%) have values
NPL_Gross           :  18/ 18 (100.0%) have values
ROA                 :  18/ 18 (100.0%) have values
BOPO                :  18/ 18 (100.0%) have values
NIM                 :  18/ 18 (100.0%) have values
LDR                 :  18/ 18 (100.0%) have values
KPMM                :  18/ 18 (100.0%) have values
Cash_Ratio          :  18/ 18 (100.0%) have values

 Ratio Statistics:


,NPL_Neto,NPL_Gross,ROA,BOPO,NIM,LDR,KPMM,Cash_Ratio
count,17,17,17,17,17,17,17,17
unique,1,1,1,1,1,1,1,1
top,,,,,,,,
freq,17,17,17,17,17,17,17,17


## 11. Top Banks by Total Aset

In [146]:
latest_year = df_export['Tahun'].max()
df_latest = df_export[df_export['Tahun'] == latest_year].copy()

print(f" Top 10 Banks by Total Aset ({latest_year}):")
df_top = df_latest.nlargest(10, 'Total_Aset')[['Nama_BPR', 'Provinsi', 'Total_Aset', 'ROA', 'NPL_Gross']]
df_top

 Top 10 Banks by Total Aset (2012):


,Nama_BPR,Provinsi,Total_Aset,ROA,NPL_Gross
6,PT Bank Perekonomian Rakyat Citra Dumoga,Provinsi Sulawesi Utara,263557393.0,,
1,PT Bank Perekonomian Rakyat Prisma Dana,Provinsi Sulawesi Utara,218021331.0,,
11,PT Bank Perekonomian Rakyat Dana Raya,Provinsi Sulawesi Utara,141497034.0,,
10,PT Bank Perekonomian Rakyat Celebes Mitra Perdana,Provinsi Sulawesi Utara,44003101.0,,
12,PT Bank Perekonomian Rakyat Kredit Mandiri Cel...,Provinsi Sulawesi Utara,36571407.0,,
0,PT Bank Perekonomian Rakyat Nusa Utara,Provinsi Sulawesi Utara,20701945.0,,
3,PT Bank Perekonomian Rakyat Millenia,Provinsi Sulawesi Utara,18917545.0,,
4,PT Bank Perekonomian Rakyat Mapalus Tumetenden,Provinsi Sulawesi Utara,15906072.0,,
14,PT Bank Perekonomian Rakyat Danaku Mapan Lestari,Provinsi Sulawesi Utara,14564097.0,,
7,PT Bank Perekonomian Rakyat Paro Laba,Provinsi Sulawesi Utara,13789176.0,,


---

## 📝 Notes:

### Key Features:
- ✅ **RATIO EXTRACTION FIXED** - Menggunakan `force_table_index=37` untuk ratio table
- ✅ **Thread-safe** - Menggunakan ThreadPoolExecutor
- ✅ **Progress tracking** - Real-time progress dengan tqdm
- ✅ **Error handling** - Robust error handling
- ✅ **Auto-save** - CSV dan Excel otomatis tersimpan

### Tips:
- Start dengan `MAX_WORKERS=4-6` untuk stability
- Untuk dataset besar, scrape per provinsi
- Check ratio values dengan cell "Data Quality Check"
- Jika timeout banyak, kurangi `MAX_WORKERS`

### Troubleshooting:
- **Ratio masih 0?** → Cek apakah table index 37 ada di tahun tersebut
- **Banyak failed?** → Kurangi MAX_WORKERS atau cek koneksi internet
- **Memory error?** → Scrape per provinsi, jangan semua sekaligus